In [ ]:
import pandas as pd
import numpy as np
import pyarrow.parquet as pq

# 메모리 부족을 피하기 위해 기본은 샘플 행만 로드합니다.
# 전체 로드가 필요하면 n_rows=None으로 변경하세요.
def load_parquet_safely(path, n_rows=200_000, batch_size=50_000):
    if n_rows is None:
        return pd.read_parquet(path, engine="pyarrow")

    parquet_file = pq.ParquetFile(path)
    batches = []
    loaded = 0

    for batch in parquet_file.iter_batches(batch_size=batch_size):
        batch_df = batch.to_pandas(types_mapper=pd.ArrowDtype)
        batches.append(batch_df)
        loaded += len(batch_df)
        if loaded >= n_rows:
            break

    if not batches:
        return pd.DataFrame()

    result = pd.concat(batches, ignore_index=True)
    return result.iloc[:n_rows].copy()

# data/ 폴더의 샘플/전체 데이터 로드
train = load_parquet_safely("../data/train_sample.parquet", n_rows=200_000)
test = load_parquet_safely("../data/test.parquet", n_rows=200_000)

print(f"train shape (sample): {train.shape}")
print(f"test shape (sample): {test.shape}")

In [ ]:
!pip install pandas numpy pyarrow

  Using cached pandas-3.0.1-cp314-cp314-win_amd64.whl.metadata (19 kB)
  Using cached tzdata-2025.3-py2.py3-none-any.whl.metadata (1.4 kB)
Using cached pandas-3.0.1-cp314-cp314-win_amd64.whl (9.9 MB)
   ---------------------------------------- 0.0/12.4 MB ? eta -:--:--
   - -------------------------------------- 0.5/12.4 MB 11.9 MB/s eta 0:00:02
   -------------- ------------------------- 4.5/12.4 MB 18.0 MB/s eta 0:00:01
   --------------------------- ------------ 8.7/12.4 MB 19.2 MB/s eta 0:00:01
   ---------------------------------------  12.3/12.4 MB 18.6 MB/s eta 0:00:01
   ---------------------------------------- 12.4/12.4 MB 17.9 MB/s  0:00:00
Using cached tzdata-2025.3-py2.py3-none-any.whl (348 kB)

   ---------------------------------------- 0/3 [tzdata]
   ---------------------------------------- 0/3 [tzdata]
   ---------------------------------------- 0/3 [tzdata]
   ---------------------------------------- 0/3 [tzdata]
   ------------- -------------------------- 1/3 [numpy]


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip
